# 7.16 — FPN & RetinaNet

Feature Pyramid Networks (FPN) give an object detector a scale-aware memory: fine grids keep small-object location detail while coarse grids carry strong semantics. RetinaNet adds focal loss so the many easy background anchors in a one-stage detector stop drowning the few foreground examples that actually teach the model.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build FPN and RetinaNet one idea at a time. Run each cell in order and inspect the printed arrays: every operation is written in NumPy so the pyramid merge, anchor assignment, focal loss, NMS, and AP calculation are visible rather than hidden inside a detection library. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, simple image-like feature maps, and detection math.
import matplotlib.pyplot as plt  # heatmaps, bars, and precision-recall plots.
np.random.seed(0)  # reproducibility for synthetic feature maps and examples.

### 1. A backbone naturally makes a scale pyramid

A convolutional backbone does not keep one resolution forever. Each later stage sees a larger piece of the image and is usually more semantic, but its grid is coarser. For small objects, that coarseness is dangerous: a 2×2 object might occupy several cells on a fine map but disappear into one cell on a coarse map. FPN starts from this fact and asks how to make every scale useful.

In [ ]:
image_w = np.zeros((8, 8))  # a tiny image grid.
image_w[1:3, 1:3] = 1.0     # small bright object in the top-left.
image_w[5:8, 4:8] = 0.7     # larger object in the bottom-right.
print("image shape:", image_w.shape, "sum:", round(float(image_w.sum()), 2))
assert image_w.shape == (8, 8)

▶ What you'll see: an 8×8 toy image with one small object and one larger object.

In [ ]:
def avg_pool2_w(x_w):
    h_w, w_w = x_w.shape
    return x_w.reshape(h_w // 2, 2, w_w // 2, 2).mean(axis=(1, 3))

c2_w = image_w.copy()        # fine stage: high spatial resolution.
c3_w = avg_pool2_w(c2_w)     # middle stage: half resolution.
c4_w = avg_pool2_w(c3_w)     # coarse stage: quarter resolution.
print("C2/C3/C4 shapes:", c2_w.shape, c3_w.shape, c4_w.shape)
print("C4:\n", np.round(c4_w, 2))

▶ What you'll see: the pyramid shrinks from 8×8 to 4×4 to 2×2; the small object becomes diluted at the coarsest level.

In [ ]:
fig_w, ax_w = plt.subplots(1, 3, figsize=(8, 2.6))
for axis_w, feat_w, title_w in zip(ax_w, [c2_w, c3_w, c4_w], ["C2 fine", "C3 middle", "C4 coarse"]):
    axis_w.imshow(feat_w, cmap="viridis", vmin=0, vmax=1)
    axis_w.set_title(title_w)
    axis_w.set_xticks([]); axis_w.set_yticks([])
plt.suptitle("1: backbone stages trade resolution for semantics"); plt.show()

▶ What you'll see: the fine map localizes the small square; the coarse map is compact but has lost sharp boundaries.

*Why it's done this way:* Detection needs both **where** and **what**. Early feature maps preserve where because their pixels/cells still line up with image detail; late feature maps know more about what because they summarize larger neighborhoods. FPN exists because neither end of that tradeoff is enough by itself.

### 2. FPN top-down addition: align, project, upsample, add

FPN builds pyramid features with the recurrence $P_l = W_l C_l + \operatorname{Up}(P_{l+1})$. The lateral term $W_l C_l$ means "project the fine backbone map to the right channel space." The top-down term means "bring coarse semantics back to this fine grid." Only after shape alignment does addition make sense.

In [ ]:
lateral_w = np.array([[1., 2.],
                      [3., 4.]])
coarse_w = np.array([[10.],
                     [20.]])
print("lateral shape:", lateral_w.shape, "coarse shape:", coarse_w.shape)

▶ What you'll see: the lateral map is 2×2, while the coarse map is 2×1 and cannot yet be added elementwise.

In [ ]:
upsampled_w = np.repeat(coarse_w, 2, axis=1)  # nearest-neighbor upsample along width.
merged_w = lateral_w + upsampled_w            # numeric FPN addition after shape alignment.
print("upsampled coarse:\n", upsampled_w)
print("merged P_l:\n", merged_w)
assert np.array_equal(merged_w, np.array([[11., 12.], [23., 24.]]))

▶ What you'll see: the merged map is `[[11, 12], [23, 24]]`, matching the lesson arithmetic.

In [ ]:
fig_w, ax_w = plt.subplots(1, 3, figsize=(8, 2.6))
for axis_w, feat_w, title_w in zip(ax_w, [lateral_w, upsampled_w, merged_w], ["lateral W_l C_l", "Up(P_{l+1})", "sum P_l"]):
    axis_w.imshow(feat_w, cmap="magma")
    axis_w.set_title(title_w)
    axis_w.set_xticks([]); axis_w.set_yticks([])
plt.suptitle("2: FPN merges semantic coarse context into a fine grid"); plt.show()

▶ What you'll see: the final map keeps the 2×2 fine layout but carries large values from the coarser semantic map.

*Why it's done this way:* Addition forces the two sources to describe the **same cell locations and channel meanings**. Upsampling transfers high-level context to a denser grid, while the lateral projection keeps location-specific evidence. Concatenating everything would increase channels and cost; addition is a cheap residual-style merge once dimensions match.

### 3. Multi-scale anchors improve the chance of a good IoU match

Dense detectors place many anchors at many locations and assign labels by intersection-over-union (IoU). FPN does not remove anchors; it gives different pyramid levels feature maps whose stride and anchor sizes better match object sizes. The target object below is `[1,1,3,3]`, and the best anchor is the one with the largest IoU.

In [ ]:
def iou_w(box1_w, box2_w):
    x1_w = max(box1_w[0], box2_w[0]); y1_w = max(box1_w[1], box2_w[1])
    x2_w = min(box1_w[2], box2_w[2]); y2_w = min(box1_w[3], box2_w[3])
    inter_w = max(0, x2_w - x1_w) * max(0, y2_w - y1_w)
    area1_w = (box1_w[2] - box1_w[0]) * (box1_w[3] - box1_w[1])
    area2_w = (box2_w[2] - box2_w[0]) * (box2_w[3] - box2_w[1])
    return inter_w / (area1_w + area2_w - inter_w)

gt_w = np.array([1., 1., 3., 3.])
anchors_w = np.array([[0., 0., 2., 2.], [0., 0., 3., 3.], [1., 1., 4., 4.]])
ious_w = np.array([iou_w(a_w, gt_w) for a_w in anchors_w])
print("IoUs:", np.round(ious_w, 3), "best index:", int(np.argmax(ious_w)))
assert np.allclose(np.round(ious_w, 3), [0.143, 0.444, 0.444])
assert int(np.argmax(ious_w)) == 1

▶ What you'll see: the tiny anchor has IoU ≈0.143, while the larger anchors reach ≈0.444; NumPy returns the first best index, 1.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["small", "large A", "large B"], ious_w, color=["gray", "teal", "teal"])
plt.axhline(0.5, color="red", linestyle="--", label="common positive threshold")
plt.ylabel("IoU with ground truth"); plt.title("3: anchor quality by scale"); plt.legend(); plt.show()

▶ What you'll see: bigger, better-aligned anchors are much closer to the ground-truth box, even if they are still below 0.5.

*Why it's done this way:* IoU is a geometric overlap fraction, so anchor size and position matter before classification even starts. FPN helps because small anchors can live on fine grids and large anchors on coarse grids, increasing the chance that some anchor has enough overlap to become useful training signal.

### 4. RetinaNet focal loss silences easy background anchors

A one-stage detector evaluates thousands of anchors. Most are background and quickly become easy: the model assigns high probability to their true background label. Cross-entropy still adds many small losses, and many small losses can dominate. Focal loss changes the weight to $\alpha(1-p_t)^\gamma$, where $p_t$ is the probability assigned to the true class.

In [ ]:
def focal_loss_w(pt_w, alpha_w=0.25, gamma_w=2.0):
    pt_w = np.clip(pt_w, 1e-9, 1.0)
    return -alpha_w * (1 - pt_w) ** gamma_w * np.log(pt_w)

def ce_loss_w(pt_w, alpha_w=0.25):
    pt_w = np.clip(pt_w, 1e-9, 1.0)
    return -alpha_w * np.log(pt_w)

pts_w = np.array([0.9, 0.1])
fl_w = focal_loss_w(pts_w)
print("focal losses easy/hard:", np.round(fl_w, 5))
print("hard/easy ratio:", round(float(fl_w[1] / fl_w[0])))
assert round(float(fl_w[0]), 5) == 0.00026
assert round(float(fl_w[1]), 3) == 0.466

▶ What you'll see: the hard example has loss about 1770× larger than the easy example.

In [ ]:
pt_grid_w = np.linspace(0.01, 0.99, 120)
plt.figure(figsize=(4.8, 3))
plt.plot(pt_grid_w, ce_loss_w(pt_grid_w), label="cross-entropy", color="gray")
plt.plot(pt_grid_w, focal_loss_w(pt_grid_w), label="focal γ=2", color="crimson")
plt.yscale("log"); plt.xlabel("p_t = probability of true class"); plt.ylabel("loss (log scale)")
plt.title("4: focal loss shrinks confident examples"); plt.legend(); plt.show()

▶ What you'll see: focal loss nearly vanishes near `p_t=1` while staying large for low true-class probability.

*Why it's done this way:* The modulating factor $(1-p_t)^\gamma$ is close to zero when the model is already correct and confident, and close to one when it is wrong. That keeps easy background anchors from winning by sheer count while preserving pressure on hard foreground and hard background examples.

### 5. Dense detectors still need NMS cleanup

FPN and focal loss improve features and training, but a dense detector still emits overlapping boxes around the same object. Non-maximum suppression (NMS) keeps the highest-scoring box first, then removes later boxes whose IoU with a kept box is too high.

In [ ]:
def iou_pair_w(a_w, b_w):
    x1_w = max(a_w[0], b_w[0]); y1_w = max(a_w[1], b_w[1])
    x2_w = min(a_w[2], b_w[2]); y2_w = min(a_w[3], b_w[3])
    inter_w = max(0, x2_w - x1_w) * max(0, y2_w - y1_w)
    area_a_w = (a_w[2] - a_w[0]) * (a_w[3] - a_w[1])
    area_b_w = (b_w[2] - b_w[0]) * (b_w[3] - b_w[1])
    return inter_w / (area_a_w + area_b_w - inter_w)

boxes_w = np.array([[0., 0., 4., 4.], [0.5, 0.5, 4.5, 4.5], [6., 6., 8., 8.]])
scores_w = np.array([0.90, 0.80, 0.70])
overlap01_w = iou_pair_w(boxes_w[0], boxes_w[1])
print("IoU box0-box1:", round(overlap01_w, 3))
assert round(overlap01_w, 3) == 0.620

▶ What you'll see: boxes 0 and 1 overlap heavily, so they probably describe the same object.

In [ ]:
keep_w = []
order_w = list(np.argsort(scores_w)[::-1])
while order_w:
    current_w = order_w.pop(0)
    keep_w.append(current_w)
    order_w = [j_w for j_w in order_w if iou_pair_w(boxes_w[current_w], boxes_w[j_w]) <= 0.3]
print("kept indices:", keep_w)
assert keep_w == [0, 2]

▶ What you'll see: NMS keeps the best duplicate (0), suppresses the overlapping duplicate (1), and keeps the separate box (2).

In [ ]:
plt.figure(figsize=(4, 4))
for idx_w, box_w in enumerate(boxes_w):
    color_w = "seagreen" if idx_w in keep_w else "crimson"
    plt.gca().add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, edgecolor=color_w, linewidth=2))
    plt.text(box_w[0], box_w[1] - 0.15, f"{idx_w}:{scores_w[idx_w]:.1f}", color=color_w)
plt.xlim(-0.5, 8.5); plt.ylim(8.5, -0.5); plt.title("5: NMS keeps green, suppresses red"); plt.show()

▶ What you'll see: one red duplicate lies almost on top of a green box, while the distant green box remains.

*Why it's done this way:* Dense prediction intentionally over-generates candidates to avoid missing objects. NMS converts that redundant set into a cleaner set by using score as confidence and IoU as duplicate geometry.

### 6. AP measures whether ranked detections are useful

Training losses are proxies. Detection quality is usually judged by ranked detections: as we lower the score threshold, recall rises and precision may fall. Average precision (AP) is the area under that precision-recall curve.

In [ ]:
precision_w = np.array([1.00, 0.75, 0.60])
recall_w = np.array([0.33, 0.67, 1.00])
recall_steps_w = np.diff(np.r_[0.0, recall_w])
ap_w = float(np.sum(precision_w * recall_steps_w))
print("recall steps:", np.round(recall_steps_w, 2))
print("AP:", round(ap_w, 3))
assert round(ap_w, 3) == 0.783

▶ What you'll see: the three rectangle areas sum to AP ≈0.783.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.step(np.r_[0, recall_w], np.r_[precision_w[0], precision_w], where="post", color="navy")
plt.fill_between(np.r_[0, recall_w], np.r_[precision_w[0], precision_w], step="post", alpha=0.2, color="navy")
plt.xlabel("recall"); plt.ylabel("precision"); plt.ylim(0, 1.05)
plt.title(f"6: AP area = {ap_w:.3f}"); plt.show()

▶ What you'll see: AP is the shaded area under the ranked precision-recall curve.

*Why it's done this way:* FPN and focal loss matter only if they improve the final ranked detections. AP rewards detectors that find objects (recall) while keeping high-scoring predictions correct (precision), which is exactly the practical detection tradeoff.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses
> small numbers, prints every intermediate with an inline `# ->` result, draws one picture,
> and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Downsampling makes a scale pyramid

Average pooling turns a fine image grid into coarser backbone stages. Watch a single bright pixel blur while a larger block survives.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_image = np.zeros((4, 4), dtype=float)
t1_image[0, 0] = 4.0
t1_image[2:4, 2:4] = 2.0
print("fine image:\n", t1_image)  # -> [[4,0,0,0],[0,0,0,0],[0,0,2,2],[0,0,2,2]]
t1_pooled = t1_image.reshape(2, 2, 2, 2).mean(axis=(1, 3))
print("2x2 pooled map:\n", t1_pooled)  # -> [[1,0],[0,2]]
t1_coarse = t1_pooled.reshape(1, 2, 1, 2).mean(axis=(1, 3))
print("coarsest map:\n", t1_coarse)  # -> [[0.75]]
assert t1_pooled.shape == (2, 2)
assert round(float(t1_coarse[0, 0]), 2) == 0.75

fig, t1_ax = plt.subplots(1, 3, figsize=(7.2, 2.4))
for t1_axis, t1_map, t1_title in zip(t1_ax, [t1_image, t1_pooled, t1_coarse], ["fine", "pooled", "coarse"]):
    t1_axis.imshow(t1_map, cmap="viridis", vmin=0, vmax=4)
    t1_axis.set_title(t1_title)
    t1_axis.set_xticks([])
    t1_axis.set_yticks([])
plt.suptitle("Toy 1 · resolution shrinks while signal pools")
plt.show()

▶ What you'll see: the fine 4×4 grid becomes a 2×2 map `[[1,0],[0,2]]`, then one coarse value `0.75`.

### ✍️ Toy 2 · FPN upsamples before adding

Top-down semantics must be copied onto the lateral grid before elementwise addition is legal.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_lateral = np.array([[1.0, 3.0], [2.0, 4.0]])
print("lateral map:\n", t2_lateral)  # -> [[1,3],[2,4]]
t2_coarse = np.array([[10.0]])
print("coarse semantic map:\n", t2_coarse)  # -> [[10]]
t2_topdown = np.repeat(np.repeat(t2_coarse, 2, axis=0), 2, axis=1)
print("upsampled top-down:\n", t2_topdown)  # -> [[10,10],[10,10]]
t2_merged = t2_lateral + t2_topdown
print("merged P level:\n", t2_merged)  # -> [[11,13],[12,14]]
assert np.array_equal(t2_merged, np.array([[11.0, 13.0], [12.0, 14.0]]))

fig, t2_ax = plt.subplots(1, 3, figsize=(7.2, 2.4))
for t2_axis, t2_map, t2_title in zip(t2_ax, [t2_lateral, t2_topdown, t2_merged], ["lateral", "top-down", "sum"]):
    t2_axis.imshow(t2_map, cmap="magma")
    t2_axis.set_title(t2_title)
    t2_axis.set_xticks([])
    t2_axis.set_yticks([])
plt.suptitle("Toy 2 · align then add")
plt.show()

▶ What you'll see: the scalar coarse context expands to 2×2 and adds to every aligned lateral cell.

### ✍️ Toy 3 · Anchor IoU picks the best scale

Dense anchors compete by geometric overlap. The best anchor is the one with the largest IoU to the ground truth.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_gt = np.array([1.0, 1.0, 3.0, 3.0])
t3_anchors = np.array([[1.0, 1.0, 2.0, 2.0], [0.5, 0.5, 3.5, 3.5], [1.0, 1.0, 3.0, 3.0]])
print("ground-truth box:", t3_gt)  # -> [1,1,3,3]
print("anchors:\n", t3_anchors)  # -> three candidate boxes

def t3_iou(t3_a, t3_b):
    t3_ix1 = max(t3_a[0], t3_b[0])
    t3_iy1 = max(t3_a[1], t3_b[1])
    t3_ix2 = min(t3_a[2], t3_b[2])
    t3_iy2 = min(t3_a[3], t3_b[3])
    t3_inter = max(0.0, t3_ix2 - t3_ix1) * max(0.0, t3_iy2 - t3_iy1)
    t3_area_a = (t3_a[2] - t3_a[0]) * (t3_a[3] - t3_a[1])
    t3_area_b = (t3_b[2] - t3_b[0]) * (t3_b[3] - t3_b[1])
    t3_union = t3_area_a + t3_area_b - t3_inter
    return t3_inter / t3_union

t3_ious = np.array([t3_iou(t3_anchor, t3_gt) for t3_anchor in t3_anchors])
print("IoUs:", np.round(t3_ious, 3))  # -> [0.25, 0.444, 1.0]
t3_best = int(np.argmax(t3_ious))
print("best anchor index:", t3_best)  # -> 2
assert t3_best == 2
assert round(float(t3_ious[1]), 3) == 0.444

plt.figure(figsize=(4.5, 3))
plt.bar(["small", "loose", "exact"], t3_ious, color=["gray", "teal", "seagreen"])
plt.ylim(0, 1.05)
plt.ylabel("IoU")
plt.title("Toy 3 · anchor quality by overlap")
plt.show()

▶ What you'll see: the exact-size anchor wins with IoU `1.0`; the loose larger anchor is only `0.444`.

### ✍️ Toy 4 · Focal loss shrinks easy anchors

The focal multiplier `(1-p_t)^γ` makes confident true-class examples nearly vanish while keeping hard anchors loud.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_pt = np.array([0.95, 0.50, 0.10])
print("true-class probabilities:", t4_pt)  # -> [0.95,0.5,0.1]
t4_alpha = 0.25
t4_gamma = 2.0
t4_ce = -t4_alpha * np.log(t4_pt)
print("alpha-weighted CE:", np.round(t4_ce, 6))  # -> [0.012823,0.173287,0.575646]
t4_mod = (1 - t4_pt) ** t4_gamma
print("focal multipliers:", np.round(t4_mod, 4))  # -> [0.0025,0.25,0.81]
t4_focal = t4_ce * t4_mod
print("focal losses:", np.round(t4_focal, 6))  # -> [0.000032,0.043322,0.466273]
assert round(float(t4_focal[0]), 6) == 0.000032
assert round(float(t4_focal[2]), 3) == 0.466

plt.figure(figsize=(4.8, 3))
plt.plot(t4_pt, t4_ce, marker="o", label="CE")
plt.plot(t4_pt, t4_focal, marker="o", label="focal")
plt.xlabel("p_t")
plt.ylabel("loss")
plt.title("Toy 4 · easy anchors get tiny loss")
plt.legend()
plt.show()

▶ What you'll see: the easy `p_t=0.95` loss drops from about `0.0128` to `0.000032`.

### ✍️ Toy 5 · NMS suppresses overlapping duplicates

Non-maximum suppression keeps the highest-score box first, then rejects lower-score boxes that overlap it too much.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_boxes = np.array([[0.0, 0.0, 2.0, 2.0], [0.2, 0.2, 2.2, 2.2], [3.0, 3.0, 4.0, 4.0]])
t5_scores = np.array([0.85, 0.80, 0.60])
print("scores:", t5_scores)  # -> [0.85,0.8,0.6]
t5_order = list(np.argsort(t5_scores)[::-1])
print("score order:", t5_order)  # -> [0,1,2]

def t5_iou(t5_a, t5_b):
    t5_ix1 = max(t5_a[0], t5_b[0])
    t5_iy1 = max(t5_a[1], t5_b[1])
    t5_ix2 = min(t5_a[2], t5_b[2])
    t5_iy2 = min(t5_a[3], t5_b[3])
    t5_inter = max(0.0, t5_ix2 - t5_ix1) * max(0.0, t5_iy2 - t5_iy1)
    t5_area_a = (t5_a[2] - t5_a[0]) * (t5_a[3] - t5_a[1])
    t5_area_b = (t5_b[2] - t5_b[0]) * (t5_b[3] - t5_b[1])
    return t5_inter / (t5_area_a + t5_area_b - t5_inter)

t5_overlap = t5_iou(t5_boxes[0], t5_boxes[1])
print("IoU box0-box1:", round(float(t5_overlap), 3))  # -> 0.681
t5_keep = []
while t5_order:
    t5_current = t5_order.pop(0)
    t5_keep.append(t5_current)
    t5_order = [t5_j for t5_j in t5_order if t5_iou(t5_boxes[t5_current], t5_boxes[t5_j]) <= 0.5]
print("kept indices:", t5_keep)  # -> [0,2]
assert t5_keep == [0, 2]

plt.figure(figsize=(4, 4))
for t5_idx, t5_box in enumerate(t5_boxes):
    t5_color = "seagreen" if t5_idx in t5_keep else "crimson"
    plt.gca().add_patch(plt.Rectangle((t5_box[0], t5_box[1]), t5_box[2] - t5_box[0], t5_box[3] - t5_box[1], fill=False, edgecolor=t5_color, linewidth=2))
    plt.text(t5_box[0], t5_box[1] - 0.08, f"{t5_idx}:{t5_scores[t5_idx]:.2f}", color=t5_color)
plt.xlim(-0.2, 4.4)
plt.ylim(4.4, -0.2)
plt.title("Toy 5 · green kept, red suppressed")
plt.show()

▶ What you'll see: box 1 overlaps box 0 by `0.681`, so the lower-score duplicate is suppressed.

### ✍️ Toy 6 · AP sums ranked precision rectangles

Average precision is a sum of precision times each recall increment after detections are ranked.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_precision = np.array([1.0, 0.8, 0.5])
print("precision values:", t6_precision)  # -> [1.0,0.8,0.5]
t6_widths = np.array([0.4, 0.3, 0.3])
print("recall widths:", t6_widths)  # -> [0.4,0.3,0.3]
t6_terms = t6_precision * t6_widths
print("AP terms:", np.round(t6_terms, 3))  # -> [0.4,0.24,0.15]
t6_ap = float(t6_terms.sum())
print("AP:", round(t6_ap, 3))  # -> 0.79
t6_recall = np.cumsum(t6_widths)
print("recall points:", np.round(t6_recall, 3))  # -> [0.4,0.7,1.0]
assert round(t6_ap, 3) == 0.790

plt.figure(figsize=(4.6, 3))
plt.step(np.r_[0.0, t6_recall], np.r_[t6_precision[0], t6_precision], where="post", color="navy")
plt.fill_between(np.r_[0.0, t6_recall], np.r_[t6_precision[0], t6_precision], step="post", alpha=0.2, color="navy")
plt.ylim(0, 1.05)
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("Toy 6 · AP area = 0.790")
plt.show()

▶ What you'll see: three rectangles with areas `0.40`, `0.24`, and `0.15` sum to AP `0.79`.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, boxes, simple pyramids, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for heatmaps, boxes, bars, and precision-recall curves.
np.random.seed(0) # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Downsample an image into backbone stages

**Goal.** Build a tiny image pyramid, because FPN begins with feature maps at several resolutions. We build it in 2 steps.

In [ ]:
image_b1 = np.zeros((8, 8)) # create a simple image-like grid.
image_b1[1:3, 1:3] = 1.0 # place a small bright object.
image_b1[5:8, 4:8] = 0.7 # place a larger dimmer object.
print("image_b1 shape:", image_b1.shape, "total energy:", round(float(image_b1.sum()), 2)) # inspect the toy image.
assert image_b1.shape == (8, 8) # verify the grid size used in this example.

▶ What you'll see: the printed shape confirms an 8×8 grid with two object-like regions.

In [ ]:
c3_b1 = image_b1.reshape(4, 2, 4, 2).mean(axis=(1, 3)) # average-pool 2x2 blocks to make a coarser feature map.
c4_b1 = c3_b1.reshape(2, 2, 2, 2).mean(axis=(1, 3)) # pool again to make the coarsest feature map.
print("C3 shape:", c3_b1.shape, "C4 shape:", c4_b1.shape) # inspect how resolution changes.
plt.figure(figsize=(6, 2.4)) # create a compact side-by-side visualization.
for idx_b1, feat_b1 in enumerate([image_b1, c3_b1, c4_b1], start=1):
    plt.subplot(1, 3, idx_b1); plt.imshow(feat_b1, cmap="viridis", vmin=0, vmax=1); plt.title(f"stage {idx_b1}"); plt.xticks([]); plt.yticks([])
plt.suptitle("Basic 1: coarser stages lose spatial detail"); plt.show() # display the pyramid.

▶ What you'll see: the small object is crisp at high resolution and blurrier after pooling.

👀 Takeaway: backbone stages already form a scale pyramid, but coarse stages trade away localization.

### Basic 2 — Upsample a coarse feature map

**Goal.** Make a coarse map match a fine map's grid, because FPN can add feature maps only after spatial alignment. We build it in 2 steps.

In [ ]:
coarse_b2 = np.array([[1., 2.], [3., 4.]]) # define a 2x2 semantic map.
print("coarse_b2:\n", coarse_b2) # inspect the values before upsampling.

▶ What you'll see: four coarse cells, each representing a larger image region.

In [ ]:
up_b2 = np.repeat(np.repeat(coarse_b2, 2, axis=0), 2, axis=1) # nearest-neighbor upsample to 4x4.
print("upsampled shape:", up_b2.shape) # inspect the aligned size.
assert up_b2.shape == (4, 4) # verify the expected spatial resolution.
plt.figure(figsize=(4, 3)); plt.imshow(up_b2, cmap="magma"); plt.title("Basic 2: nearest upsampled map"); plt.colorbar(); plt.show() # visualize repeated cells.

▶ What you'll see: each coarse cell expands into a 2×2 block on the fine grid.

👀 Takeaway: upsampling copies coarse semantic context onto more spatial positions.

### Basic 3 — Add lateral and top-down maps

**Goal.** Reproduce the FPN addition formula numerically, because $P_l=W_lC_l+Up(P_{l+1})$ is just aligned array addition. We build it in 2 steps.

In [ ]:
lateral_b3 = np.array([[1., 2.], [3., 4.]]) # define the projected fine map W_l C_l.
topdown_b3 = np.array([[10., 10.], [20., 20.]]) # define the upsampled coarse map Up(P_{l+1}).
print("lateral:\n", lateral_b3, "\ntop-down:\n", topdown_b3) # inspect both aligned maps.

▶ What you'll see: both maps are 2×2, so elementwise addition is legal.

In [ ]:
pyramid_b3 = lateral_b3 + topdown_b3 # merge location detail with semantic context.
print("P_l:\n", pyramid_b3) # inspect the merged feature map.
assert np.array_equal(pyramid_b3, np.array([[11., 12.], [23., 24.]])) # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)); plt.imshow(pyramid_b3, cmap="viridis"); plt.colorbar(label="merged value"); plt.title("Basic 3: FPN addition"); plt.show() # visualize the merged map.

▶ What you'll see: each output cell equals the lateral value plus the semantic top-down value.

👀 Takeaway: FPN addition preserves the fine grid while injecting coarse semantic information.

### Basic 4 — Compute one box IoU

**Goal.** Measure anchor quality with intersection-over-union, because detectors assign positives and negatives from geometric overlap. We build it in 2 steps.

In [ ]:
box_a_b4 = np.array([0., 0., 2., 2.]) # define a small anchor box as x1,y1,x2,y2.
box_g_b4 = np.array([1., 1., 3., 3.]) # define the ground-truth box.
inter_w_b4 = max(0, min(box_a_b4[2], box_g_b4[2]) - max(box_a_b4[0], box_g_b4[0])) # compute intersection width.
inter_h_b4 = max(0, min(box_a_b4[3], box_g_b4[3]) - max(box_a_b4[1], box_g_b4[1])) # compute intersection height.
print("intersection width/height:", inter_w_b4, inter_h_b4) # inspect overlap dimensions.

▶ What you'll see: the two boxes overlap by a 1×1 square.

In [ ]:
inter_b4 = inter_w_b4 * inter_h_b4 # compute intersection area.
area_a_b4 = (box_a_b4[2] - box_a_b4[0]) * (box_a_b4[3] - box_a_b4[1]) # compute anchor area.
area_g_b4 = (box_g_b4[2] - box_g_b4[0]) * (box_g_b4[3] - box_g_b4[1]) # compute ground-truth area.
iou_b4 = inter_b4 / (area_a_b4 + area_g_b4 - inter_b4) # divide intersection by union.
print("IoU:", round(float(iou_b4), 3)) # inspect the overlap score.
assert round(float(iou_b4), 3) == 0.143 # verify 1 / 7.
plt.figure(figsize=(4, 4)); plt.gca().add_patch(plt.Rectangle((0, 0), 2, 2, fill=False, edgecolor="teal", linewidth=2)); plt.gca().add_patch(plt.Rectangle((1, 1), 2, 2, fill=False, edgecolor="orange", linewidth=2)); plt.xlim(-.2, 3.2); plt.ylim(3.2, -.2); plt.title("Basic 4: overlapping boxes"); plt.show() # show the geometry.

▶ What you'll see: a small overlap compared with the total covered area.

👀 Takeaway: IoU converts box alignment into a number between 0 and 1.

### Basic 5 — Choose the best anchor

**Goal.** Pick the anchor with the highest IoU, because dense detectors need a rule for assigning each object to candidate boxes. We build it in 2 steps.

In [ ]:
gt_b5 = np.array([1., 1., 3., 3.]) # define one ground-truth object box.
anchors_b5 = np.array([[0., 0., 2., 2.], [0., 0., 3., 3.], [1., 1., 4., 4.]]) # define anchors from different locations/scales.
print("anchors_b5 shape:", anchors_b5.shape) # inspect the candidate set.

▶ What you'll see: three candidate anchors compete to explain the same object.

In [ ]:
ious_b5 = [] # store IoUs for each candidate anchor.
for a_b5 in anchors_b5:
    ix1_b5, iy1_b5 = max(a_b5[0], gt_b5[0]), max(a_b5[1], gt_b5[1]) # intersection top-left.
    ix2_b5, iy2_b5 = min(a_b5[2], gt_b5[2]), min(a_b5[3], gt_b5[3]) # intersection bottom-right.
    inter_b5 = max(0, ix2_b5 - ix1_b5) * max(0, iy2_b5 - iy1_b5) # intersection area.
    union_b5 = (a_b5[2]-a_b5[0])*(a_b5[3]-a_b5[1]) + 4 - inter_b5 # union with the 2x2 ground truth.
    ious_b5.append(inter_b5 / union_b5) # append IoU.
ious_b5 = np.array(ious_b5) # convert to array for argmax and plotting.
print("IoUs:", np.round(ious_b5, 3), "best:", int(np.argmax(ious_b5))) # inspect assignment.
assert int(np.argmax(ious_b5)) == 1 # verify first best anchor index.
plt.figure(figsize=(4, 3)); plt.bar(["A0", "A1", "A2"], ious_b5, color="slateblue"); plt.title("Basic 5: anchor IoU scores"); plt.ylabel("IoU"); plt.show() # compare anchors.

▶ What you'll see: anchors 1 and 2 tie, and the first maximum is selected.

👀 Takeaway: multi-scale anchors help only if at least one candidate overlaps the object well.

### Basic 6 — Compute cross-entropy for true-class probability

**Goal.** Use $p_t$ correctly, because both cross-entropy and focal loss depend on the probability assigned to the true class. We build it in 2 steps.

In [ ]:
pt_b6 = np.array([0.9, 0.1]) # define an easy true-class probability and a hard one.
ce_b6 = -np.log(pt_b6) # compute binary/multiclass true-class cross-entropy pieces.
print("cross-entropy:", np.round(ce_b6, 3)) # inspect losses before focal weighting.
assert round(float(ce_b6[0]), 3) == 0.105 # verify -log(0.9).

▶ What you'll see: the hard example with `p_t=0.1` has much larger cross-entropy.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["easy p_t=.9", "hard p_t=.1"], ce_b6, color=["teal", "crimson"]); plt.title("Basic 6: cross-entropy uses p_t"); plt.ylabel("-log(p_t)"); plt.xticks(rotation=10); plt.show() # visualize the gap.

▶ What you'll see: the hard example is already larger before any focal weighting.

👀 Takeaway: $p_t$ always means the model probability for the correct label, not always the foreground probability.

### Basic 7 — Add the focal modulating factor

**Goal.** Down-weight easy examples with $(1-p_t)^\gamma$, because RetinaNet must handle many easy background anchors. We build it in 2 steps.

In [ ]:
pt_b7 = np.array([0.9, 0.1]) # define easy and hard true-class probabilities.
gamma_b7 = 2.0 # use RetinaNet's common focal exponent.
mod_b7 = (1 - pt_b7) ** gamma_b7 # compute the focal modulating factors.
print("modulating factors:", np.round(mod_b7, 3)) # inspect how confidence changes weights.
assert np.allclose(np.round(mod_b7, 3), [0.01, 0.81]) # verify easy is almost silenced.

▶ What you'll see: the easy example receives a 0.01 multiplier, while the hard example receives 0.81.

In [ ]:
alpha_b7 = 0.25 # choose the common foreground balancing weight.
fl_b7 = -alpha_b7 * mod_b7 * np.log(pt_b7) # compute focal loss values.
print("focal loss:", np.round(fl_b7, 5)) # inspect final focal losses.
assert round(float(fl_b7[1]), 3) == 0.466 # verify hard-example loss.
plt.figure(figsize=(4, 3)); plt.bar(["easy", "hard"], fl_b7, color=["gray", "red"]); plt.title("Basic 7: focal loss values"); plt.ylabel("loss"); plt.show() # visualize the imbalance correction.

▶ What you'll see: the hard anchor dominates the plotted loss.

👀 Takeaway: focal loss does not delete easy examples; it makes confident ones mathematically tiny.

### Basic 8 — Count easy background dominance

**Goal.** See why many easy anchors matter, because thousands of small losses can overwhelm rare object anchors. We build it in 2 steps.

In [ ]:
n_easy_b8 = 1000 # simulate many background anchors the model already classifies well.
pt_easy_b8 = np.full(n_easy_b8, 0.95) # true-class probabilities for easy background.
pt_hard_b8 = np.array([0.2, 0.15, 0.1]) # a few hard anchors.
ce_total_b8 = float(np.sum(-np.log(pt_easy_b8)) + np.sum(-np.log(pt_hard_b8))) # total cross-entropy.
print("CE total:", round(ce_total_b8, 2)) # inspect how easy anchors accumulate.

▶ What you'll see: many easy examples still add a large total cross-entropy.

In [ ]:
fl_easy_b8 = -0.25 * (1 - pt_easy_b8) ** 2 * np.log(pt_easy_b8) # focal loss for many easy anchors.
fl_hard_b8 = -0.25 * (1 - pt_hard_b8) ** 2 * np.log(pt_hard_b8) # focal loss for few hard anchors.
print("focal easy total:", round(float(fl_easy_b8.sum()), 3), "focal hard total:", round(float(fl_hard_b8.sum()), 3)) # compare groups.
assert float(fl_hard_b8.sum()) > float(fl_easy_b8.sum()) # verify hard anchors dominate under focal loss.
plt.figure(figsize=(4, 3)); plt.bar(["1000 easy", "3 hard"], [fl_easy_b8.sum(), fl_hard_b8.sum()], color=["gray", "crimson"]); plt.title("Basic 8: focal group totals"); plt.ylabel("total focal loss"); plt.show() # visualize the corrected weighting.

▶ What you'll see: focal loss lets a few hard anchors outweigh a thousand confident easy anchors.

👀 Takeaway: RetinaNet fixes imbalance by changing loss scale, not by changing the detector into two stages.

### Basic 9 — Suppress duplicate detections with NMS

**Goal.** Remove overlapping lower-score boxes, because dense prediction produces duplicate candidates around the same object. We build it in 2 steps.

In [ ]:
boxes_b9 = np.array([[0., 0., 4., 4.], [0.5, 0.5, 4.5, 4.5], [6., 6., 8., 8.]]) # define candidate boxes.
scores_b9 = np.array([0.9, 0.8, 0.7]) # define confidence scores.
print("score order:", np.argsort(scores_b9)[::-1]) # inspect the order NMS will process.

▶ What you'll see: NMS starts from the highest-scoring candidate.

In [ ]:
def iou_b9(a_b9, b_b9):
    x1_b9, y1_b9 = max(a_b9[0], b_b9[0]), max(a_b9[1], b_b9[1])
    x2_b9, y2_b9 = min(a_b9[2], b_b9[2]), min(a_b9[3], b_b9[3])
    inter_b9 = max(0, x2_b9 - x1_b9) * max(0, y2_b9 - y1_b9)
    area_a_b9 = (a_b9[2]-a_b9[0])*(a_b9[3]-a_b9[1]); area_b_b9 = (b_b9[2]-b_b9[0])*(b_b9[3]-b_b9[1])
    return inter_b9 / (area_a_b9 + area_b_b9 - inter_b9)
keep_b9 = [0] + [j_b9 for j_b9 in [1, 2] if iou_b9(boxes_b9[0], boxes_b9[j_b9]) <= 0.3] # keep boxes not too close to box 0.
print("kept boxes:", keep_b9) # inspect NMS result.
assert keep_b9 == [0, 2] # verify duplicate suppression.
plt.figure(figsize=(4, 3)); plt.bar(["box0", "box1", "box2"], [1 if i in keep_b9 else 0 for i in range(3)], color=["green", "red", "green"]); plt.title("Basic 9: kept by NMS?"); plt.ylim(0, 1.2); plt.show() # show kept status.

▶ What you'll see: the overlapping duplicate box is suppressed while the distant box remains.

👀 Takeaway: NMS is the geometric cleanup step after dense RetinaNet predictions.

### Basic 10 — Compute a tiny AP

**Goal.** Calculate average precision from ranked detections, because detector training is judged by precision-recall behavior. We build it in 2 steps.

In [ ]:
precision_b10 = np.array([1.0, 0.75, 0.60]) # define precision after each detection threshold.
recall_b10 = np.array([0.33, 0.67, 1.0]) # define matching recall values.
steps_b10 = np.diff(np.r_[0.0, recall_b10]) # compute recall increments.
print("recall increments:", np.round(steps_b10, 2)) # inspect AP widths.

▶ What you'll see: recall increases by about one third at each point.

In [ ]:
ap_b10 = float(np.sum(precision_b10 * steps_b10)) # compute rectangular AP area.
print("AP:", round(ap_b10, 3)) # inspect final average precision.
assert round(ap_b10, 3) == 0.783 # verify the lesson AP number.
plt.figure(figsize=(4, 3)); plt.step(np.r_[0, recall_b10], np.r_[precision_b10[0], precision_b10], where="post"); plt.ylim(0, 1.05); plt.title("Basic 10: precision-recall curve"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show() # visualize ranked quality.

▶ What you'll see: the AP is the area under a stair-step precision-recall curve.

👀 Takeaway: better features and losses matter because they should improve ranked detection quality.

## 🟡 Easy

### Easy 1 — Build a full toy FPN pyramid

**Goal.** Merge C4 into C3 and C3 into C2, because FPN repeats top-down addition across levels. We build it in 3 steps.

In [ ]:
c2_e1 = np.zeros((8, 8)); c2_e1[1:3, 1:3] = 1.0; c2_e1[5:8, 4:8] = 0.7 # define a fine feature map.
c3_e1 = c2_e1.reshape(4, 2, 4, 2).mean(axis=(1, 3)) # create the middle backbone stage.
c4_e1 = c3_e1.reshape(2, 2, 2, 2).mean(axis=(1, 3)) # create the coarse backbone stage.
print("C shapes:", c2_e1.shape, c3_e1.shape, c4_e1.shape) # inspect feature scales.

▶ What you'll see: the backbone stages form 8×8, 4×4, and 2×2 maps.

In [ ]:
p4_e1 = c4_e1 # use identity projection at the coarsest level for this toy FPN.
p3_e1 = c3_e1 + np.repeat(np.repeat(p4_e1, 2, axis=0), 2, axis=1) # merge coarse semantics into C3 resolution.
p2_e1 = c2_e1 + np.repeat(np.repeat(p3_e1, 2, axis=0), 2, axis=1) # merge middle semantics into C2 resolution.
print("P sums:", round(float(p2_e1.sum()), 2), round(float(p3_e1.sum()), 2), round(float(p4_e1.sum()), 2)) # inspect signal propagation.
assert p2_e1.shape == (8, 8) and p3_e1.shape == (4, 4) and p4_e1.shape == (2, 2) # verify pyramid shapes.

▶ What you'll see: all P-levels exist, with semantic signal propagated to fine grids.

In [ ]:
fig_e1, ax_e1 = plt.subplots(1, 3, figsize=(8, 2.6)) # create one figure for the pyramid.
for axis_e1, feat_e1, title_e1 in zip(ax_e1, [p2_e1, p3_e1, p4_e1], ["P2", "P3", "P4"]):
    axis_e1.imshow(feat_e1, cmap="viridis"); axis_e1.set_title(title_e1); axis_e1.set_xticks([]); axis_e1.set_yticks([])
plt.suptitle("Easy 1: top-down FPN pyramid"); plt.show() # visualize all FPN levels.

▶ What you'll see: P2 keeps the finest grid while receiving context from P3 and P4.

👀 Takeaway: FPN is a repeated align-and-add pathway from coarse semantic maps back to fine grids.

### Easy 2 — Assign objects to pyramid levels by size

**Goal.** Route boxes to reasonable feature levels, because small objects should be detected on finer grids and large objects on coarser grids. We build it in 3 steps.

In [ ]:
boxes_e2 = np.array([[1., 1., 3., 3.], [0., 0., 6., 6.], [2., 2., 10., 14.]]) # define small, medium, and large boxes.
sizes_e2 = np.sqrt((boxes_e2[:, 2] - boxes_e2[:, 0]) * (boxes_e2[:, 3] - boxes_e2[:, 1])) # compute square-root area scale.
print("box sizes:", np.round(sizes_e2, 2)) # inspect scale as one number per box.

▶ What you'll see: the boxes have increasing geometric sizes.

In [ ]:
levels_e2 = np.where(sizes_e2 < 4, "P2", np.where(sizes_e2 < 8, "P3", "P4")) # route small/medium/large boxes to levels.
print("assigned levels:", levels_e2) # inspect scale-aware assignments.
assert list(levels_e2) == ["P2", "P3", "P4"] # verify the routing rule.

▶ What you'll see: the smallest object goes to P2, the middle to P3, and the largest to P4.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.bar(["box0", "box1", "box2"], sizes_e2, color=["teal", "orange", "purple"]); plt.axhline(4, color="gray", linestyle="--"); plt.axhline(8, color="gray", linestyle="--"); plt.title("Easy 2: size-to-level routing"); plt.ylabel("sqrt(area)"); plt.show() # visualize thresholds.

▶ What you'll see: horizontal thresholds split object sizes into pyramid levels.

👀 Takeaway: FPN gives the detector scale-specific feature maps instead of forcing every object onto one grid.

### Easy 3 — Compare focal loss to cross-entropy on many anchors

**Goal.** Quantify how focal loss changes the aggregate training signal, because class imbalance is a sum-over-anchors problem. We build it in 3 steps.

In [ ]:
pt_e3 = np.r_[np.full(500, 0.97), np.array([0.25, 0.18, 0.12, 0.08])] # simulate many easy anchors plus four hard anchors.
labels_e3 = np.r_[np.zeros(500), np.ones(4)] # mark groups only for plotting.
print("anchors:", len(pt_e3), "hard count:", int(labels_e3.sum())) # inspect the imbalance.

▶ What you'll see: four hard anchors are vastly outnumbered.

In [ ]:
ce_e3 = -0.25 * np.log(pt_e3) # alpha-scaled cross-entropy.
fl_e3 = -0.25 * (1 - pt_e3) ** 2 * np.log(pt_e3) # focal loss with gamma=2.
print("CE hard share:", round(float(ce_e3[labels_e3 == 1].sum() / ce_e3.sum()), 3)) # inspect hard share under CE.
print("FL hard share:", round(float(fl_e3[labels_e3 == 1].sum() / fl_e3.sum()), 3)) # inspect hard share under focal loss.
assert fl_e3[labels_e3 == 1].sum() / fl_e3.sum() > 0.95 # verify focal loss concentrates on hard anchors.

▶ What you'll see: hard anchors become almost all of the focal-loss total.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.bar(["CE easy", "CE hard", "FL easy", "FL hard"], [ce_e3[labels_e3==0].sum(), ce_e3[labels_e3==1].sum(), fl_e3[labels_e3==0].sum(), fl_e3[labels_e3==1].sum()], color=["gray", "red", "lightgray", "crimson"]); plt.yscale("log"); plt.title("Easy 3: aggregate loss by group"); plt.ylabel("total loss, log scale"); plt.xticks(rotation=15); plt.show() # compare total contributions.

▶ What you'll see: focal loss collapses the easy-anchor total on a log-scale plot.

👀 Takeaway: focal loss solves imbalance by making the loss budget focus on examples that are still hard.

### Easy 4 — Run NMS on sorted detections

**Goal.** Implement a complete tiny NMS loop, because RetinaNet emits many scored boxes per image. We build it in 3 steps.

In [ ]:
boxes_e4 = np.array([[0., 0., 4., 4.], [0.4, 0.4, 4.4, 4.4], [5., 5., 7., 7.], [5.2, 5.1, 7.2, 7.1]]) # define two duplicate groups.
scores_e4 = np.array([0.95, 0.85, 0.75, 0.60]) # define scores for all boxes.
print("processing order:", np.argsort(scores_e4)[::-1]) # inspect descending score order.

▶ What you'll see: higher-confidence boxes are considered before lower-confidence duplicates.

In [ ]:
def iou_e4(a_e4, b_e4):
    x1_e4, y1_e4 = max(a_e4[0], b_e4[0]), max(a_e4[1], b_e4[1]); x2_e4, y2_e4 = min(a_e4[2], b_e4[2]), min(a_e4[3], b_e4[3])
    inter_e4 = max(0, x2_e4-x1_e4) * max(0, y2_e4-y1_e4)
    area_a_e4 = (a_e4[2]-a_e4[0])*(a_e4[3]-a_e4[1]); area_b_e4 = (b_e4[2]-b_e4[0])*(b_e4[3]-b_e4[1])
    return inter_e4 / (area_a_e4 + area_b_e4 - inter_e4)
keep_e4 = [] # store kept detection indices.
order_e4 = list(np.argsort(scores_e4)[::-1]) # process from high to low score.
while order_e4:
    current_e4 = order_e4.pop(0); keep_e4.append(current_e4) # keep the current best box.
    order_e4 = [j_e4 for j_e4 in order_e4 if iou_e4(boxes_e4[current_e4], boxes_e4[j_e4]) <= 0.5] # suppress close duplicates.
print("kept:", keep_e4) # inspect final NMS choices.
assert keep_e4 == [0, 2] # verify one box per duplicate group.

▶ What you'll see: one representative survives for each object-like cluster.

In [ ]:
plt.figure(figsize=(4, 4)) # draw kept and suppressed boxes.
for i_e4, b_e4 in enumerate(boxes_e4):
    color_e4 = "green" if i_e4 in keep_e4 else "red"
    plt.gca().add_patch(plt.Rectangle((b_e4[0], b_e4[1]), b_e4[2]-b_e4[0], b_e4[3]-b_e4[1], fill=False, edgecolor=color_e4, linewidth=2))
    plt.text(b_e4[0], b_e4[1]-0.1, str(i_e4), color=color_e4)
plt.xlim(-.5, 8); plt.ylim(8, -.5); plt.title("Easy 4: NMS result"); plt.show() # visualize results.

▶ What you'll see: red boxes are lower-scoring duplicates of green boxes.

👀 Takeaway: NMS turns dense overlapping predictions into a smaller set of final detections.

### Easy 5 — Compute AP from scored detections

**Goal.** Build precision and recall from ranked true/false positives, because AP depends on detection order. We build it in 3 steps.

In [ ]:
scores_e5 = np.array([0.95, 0.90, 0.60, 0.40]) # define ranked detection scores.
tp_e5 = np.array([1, 0, 1, 1]) # mark whether each ranked detection is a true positive.
num_gt_e5 = 3 # define how many ground-truth objects exist.
order_e5 = np.argsort(scores_e5)[::-1] # sort detections by confidence.
print("rank order:", order_e5) # inspect ranking.

▶ What you'll see: detections are evaluated from highest confidence to lowest.

In [ ]:
tp_sorted_e5 = tp_e5[order_e5] # align true-positive flags with ranked detections.
precision_e5 = np.cumsum(tp_sorted_e5) / (np.arange(len(tp_sorted_e5)) + 1) # compute precision at each rank.
recall_e5 = np.cumsum(tp_sorted_e5) / num_gt_e5 # compute recall at each rank.
print("precision:", np.round(precision_e5, 3), "recall:", np.round(recall_e5, 3)) # inspect PR sequence.

▶ What you'll see: precision dips when a false positive appears, while recall only rises on true positives.

In [ ]:
ap_e5 = float(np.sum(precision_e5 * np.diff(np.r_[0.0, recall_e5]))) # compute rectangular AP.
print("AP:", round(ap_e5, 3)) # inspect ranked quality.
assert round(ap_e5, 3) == 0.806 # verify this tiny AP calculation.
plt.figure(figsize=(4, 3)); plt.step(np.r_[0, recall_e5], np.r_[precision_e5[0], precision_e5], where="post", color="navy"); plt.ylim(0, 1.05); plt.title("Easy 5: ranked AP"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show() # visualize AP curve.

▶ What you'll see: the false positive lowers the middle of the precision-recall curve.

👀 Takeaway: AP rewards high-scoring true positives and penalizes false positives that appear early.

## 🔴 Advanced

### Advanced 1 — Show why lateral projection must align channels

**Goal.** Project a multi-channel backbone map to the pyramid channel count, because FPN addition requires matching height, width, and channels. We build it in 4 steps.

In [ ]:
c_l_a1 = np.arange(2 * 2 * 3, dtype=float).reshape(2, 2, 3) # create a 2x2 feature map with 3 backbone channels.
w_l_a1 = np.array([[1., 0.], [0., 1.], [1., -1.]]) # define a 1x1 projection from 3 channels to 2 channels.
top_a1 = np.ones((2, 2, 2)) * 10 # define an upsampled top-down map with 2 pyramid channels.
print("C shape:", c_l_a1.shape, "W shape:", w_l_a1.shape, "top shape:", top_a1.shape) # inspect dimensions.

▶ What you'll see: the backbone has 3 channels but the pyramid path expects 2.

In [ ]:
lat_a1 = c_l_a1 @ w_l_a1 # apply the same 1x1 linear projection at each spatial cell.
print("lateral projected shape:", lat_a1.shape) # inspect channel alignment.
assert lat_a1.shape == top_a1.shape # verify addition is now legal.

▶ What you'll see: the lateral map now matches the top-down map's 2-channel shape.

In [ ]:
p_a1 = lat_a1 + top_a1 # perform FPN addition after channel alignment.
print("first cell lateral:", lat_a1[0, 0], "merged:", p_a1[0, 0]) # inspect a single cell.
assert np.allclose(p_a1[0, 0], [12., 9.]) # verify the concrete projected-plus-topdown values.

▶ What you'll see: the first cell combines projected local evidence with semantic context.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(p_a1[:, :, 0], cmap="viridis"); plt.colorbar(label="channel 0"); plt.title("Advanced 1: merged P_l channel 0"); plt.show() # visualize one channel of the merged feature.

▶ What you'll see: one pyramid channel as a spatial heatmap after projection and addition.

👀 Takeaway: FPN addition is simple only after a 1×1 projection makes channel meanings compatible.

### Advanced 2 — Sweep gamma in focal loss

**Goal.** See how $\gamma$ controls easy-example suppression, because focal loss has a tunable focusing strength. We build it in 3 steps.

In [ ]:
pt_a2 = np.array([0.95, 0.7, 0.3, 0.05]) # define very easy, medium, hard, and very hard examples.
gammas_a2 = np.array([0., 1., 2., 4.]) # gamma=0 is alpha-scaled cross-entropy.
print("p_t values:", pt_a2) # inspect examples being weighted.

▶ What you'll see: examples span confident-correct to badly wrong.

In [ ]:
losses_a2 = [] # store a row of losses per gamma.
for gamma_a2 in gammas_a2:
    losses_a2.append(-0.25 * (1 - pt_a2) ** gamma_a2 * np.log(pt_a2)) # compute focal loss for this gamma.
losses_a2 = np.array(losses_a2) # convert to matrix for printing and plotting.
print("loss matrix rows gamma 0,1,2,4:\n", np.round(losses_a2, 4)) # inspect focusing effect.
assert losses_a2[-1, 0] < losses_a2[0, 0] / 10000 # verify easy example is crushed at gamma=4.

▶ What you'll see: increasing gamma mostly shrinks the high-`p_t` easy example.

In [ ]:
plt.figure(figsize=(5, 3)) # create a gamma sweep plot.
for idx_a2, pt_val_a2 in enumerate(pt_a2):
    plt.plot(gammas_a2, losses_a2[:, idx_a2], marker="o", label=f"p_t={pt_val_a2}") # plot loss versus gamma for one example.
plt.yscale("log"); plt.xlabel("gamma"); plt.ylabel("loss, log scale"); plt.title("Advanced 2: focal gamma sweep"); plt.legend(); plt.show() # display focusing curves.

▶ What you'll see: easy examples fall fastest as gamma grows, while hard examples remain visible.

👀 Takeaway: larger gamma focuses training more aggressively on examples the model still gets wrong.

### Advanced 3 — Compare anchor coverage with and without multiple scales

**Goal.** Measure best IoU for objects under one anchor size versus multiple anchor sizes, because FPN is useful only when scale coverage improves. We build it in 4 steps.

In [ ]:
objects_a3 = np.array([[1., 1., 3., 3.], [1., 1., 5., 5.], [1., 1., 9., 9.]]) # define small, medium, large square objects.
single_anchor_a3 = np.array([[1., 1., 5., 5.]]) # use only one medium anchor.
multi_anchors_a3 = np.array([[1., 1., 3., 3.], [1., 1., 5., 5.], [1., 1., 9., 9.]]) # use scale-matched anchors.
print("object side lengths:", objects_a3[:, 2] - objects_a3[:, 0]) # inspect scales.

▶ What you'll see: the target objects span three sizes.

In [ ]:
def iou_a3(a_a3, b_a3):
    inter_a3 = max(0, min(a_a3[2], b_a3[2])-max(a_a3[0], b_a3[0])) * max(0, min(a_a3[3], b_a3[3])-max(a_a3[1], b_a3[1]))
    area_a_a3 = (a_a3[2]-a_a3[0])*(a_a3[3]-a_a3[1]); area_b_a3 = (b_a3[2]-b_a3[0])*(b_a3[3]-b_a3[1])
    return inter_a3 / (area_a_a3 + area_b_a3 - inter_a3)
single_best_a3 = np.array([max(iou_a3(obj_a3, anc_a3) for anc_a3 in single_anchor_a3) for obj_a3 in objects_a3]) # best IoU with one scale.
multi_best_a3 = np.array([max(iou_a3(obj_a3, anc_a3) for anc_a3 in multi_anchors_a3) for obj_a3 in objects_a3]) # best IoU with multiple scales.
print("single best:", np.round(single_best_a3, 3), "multi best:", np.round(multi_best_a3, 3)) # inspect coverage.
assert np.allclose(multi_best_a3, [1., 1., 1.]) # verify matched scales cover perfectly in this toy.

▶ What you'll see: multiple scale anchors cover all object sizes, while the single medium anchor misses small and large scales.

In [ ]:
coverage_single_a3 = float(np.mean(single_best_a3 >= 0.5)) # count objects that meet a common positive threshold.
coverage_multi_a3 = float(np.mean(multi_best_a3 >= 0.5)) # count coverage with multi-scale anchors.
print("coverage single/multi:", coverage_single_a3, coverage_multi_a3) # inspect threshold coverage.
assert coverage_multi_a3 > coverage_single_a3 # verify multi-scale improves coverage.

▶ What you'll see: multi-scale anchors have higher positive-anchor coverage.

In [ ]:
x_a3 = np.arange(3); plt.figure(figsize=(5, 3)); plt.bar(x_a3 - .18, single_best_a3, width=.36, label="one scale", color="gray"); plt.bar(x_a3 + .18, multi_best_a3, width=.36, label="multi-scale", color="teal"); plt.axhline(.5, color="red", linestyle="--"); plt.xticks(x_a3, ["small", "medium", "large"]); plt.ylabel("best IoU"); plt.title("Advanced 3: anchor scale coverage"); plt.legend(); plt.show() # visualize coverage.

▶ What you'll see: the multi-scale bars reach 1.0 for every object size.

👀 Takeaway: FPN and anchors work together: pyramid levels are valuable when anchors at those levels match object scale.

### Advanced 4 — Show how noisy hard labels can dominate focal loss

**Goal.** Inspect a focal-loss pitfall, because mislabeled hard examples receive very large weights. We build it in 3 steps.

In [ ]:
pt_clean_a4 = np.full(20, 0.9) # many clean easy examples.
pt_noisy_a4 = np.array([0.02]) # one likely mislabeled example that the model strongly disagrees with.
print("clean count:", len(pt_clean_a4), "noisy hard count:", len(pt_noisy_a4)) # inspect the imbalance.

▶ What you'll see: there is only one hard suspicious point among many clean easy points.

In [ ]:
loss_clean_a4 = -0.25 * (1 - pt_clean_a4) ** 2 * np.log(pt_clean_a4) # focal loss for clean easy examples.
loss_noisy_a4 = -0.25 * (1 - pt_noisy_a4) ** 2 * np.log(pt_noisy_a4) # focal loss for the noisy hard example.
print("clean total:", round(float(loss_clean_a4.sum()), 4), "noisy loss:", round(float(loss_noisy_a4.sum()), 4)) # compare contribution.
assert float(loss_noisy_a4.sum()) > float(loss_clean_a4.sum()) * 50 # verify domination by one hard point.

▶ What you'll see: one extremely hard example outweighs many easy examples.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.bar(["20 clean easy", "1 noisy hard"], [loss_clean_a4.sum(), loss_noisy_a4.sum()], color=["gray", "crimson"]); plt.title("Advanced 4: focal loss can emphasize noise"); plt.ylabel("total focal loss"); plt.show() # visualize the pitfall.

▶ What you'll see: the noisy hard label dominates the loss budget.

👀 Takeaway: focal loss is powerful for imbalance, but label noise can look exactly like a hard example.

### Advanced 5 — Track AP separately for small and large objects

**Goal.** Split AP by scale, because FPN's gains can be hidden when aggregate AP is dominated by large objects. We build it in 4 steps.

In [ ]:
scale_a5 = np.array(["small", "small", "large", "large", "large"]) # mark each detection's object scale group.
scores_a5 = np.array([0.92, 0.55, 0.88, 0.70, 0.40]) # define ranked detection scores.
tp_a5 = np.array([1, 0, 1, 1, 0]) # define true-positive flags after matching.
num_gt_a5 = {"small": 2, "large": 2} # define ground-truth counts per scale.
print("detections by scale:", {s_a5: int(np.sum(scale_a5 == s_a5)) for s_a5 in ["small", "large"]}) # inspect split sizes.

▶ What you'll see: detections are grouped into small-object and large-object subsets.

In [ ]:
def ap_for_scale_a5(name_a5):
    mask_a5 = scale_a5 == name_a5 # keep detections from one scale group.
    order_a5 = np.argsort(scores_a5[mask_a5])[::-1] # rank that group's detections by score.
    tp_sorted_a5 = tp_a5[mask_a5][order_a5] # read true positives in ranked order.
    precision_a5 = np.cumsum(tp_sorted_a5) / (np.arange(len(tp_sorted_a5)) + 1) # compute precision by rank.
    recall_a5 = np.cumsum(tp_sorted_a5) / num_gt_a5[name_a5] # compute recall by rank for this scale.
    return float(np.sum(precision_a5 * np.diff(np.r_[0.0, recall_a5]))), precision_a5, recall_a5 # return AP and curve.
ap_small_a5, p_small_a5, r_small_a5 = ap_for_scale_a5("small") # compute AP for small objects.
ap_large_a5, p_large_a5, r_large_a5 = ap_for_scale_a5("large") # compute AP for large objects.
print("AP small/large:", round(ap_small_a5, 3), round(ap_large_a5, 3)) # inspect scale-specific quality.
assert round(ap_small_a5, 3) == 0.5 and round(ap_large_a5, 3) == 1.0 # verify scale AP values.

▶ What you'll see: large-object AP is perfect in this toy, while small-object AP is lower.

In [ ]:
ap_all_a5 = float(np.mean([ap_small_a5, ap_large_a5])) # compute a simple macro average across scales.
print("macro AP:", round(ap_all_a5, 3)) # inspect aggregate after scale split.
assert round(ap_all_a5, 3) == 0.75 # verify macro average.

▶ What you'll see: the aggregate hides the fact that small objects are the weak group.

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["small AP", "large AP", "macro"], [ap_small_a5, ap_large_a5, ap_all_a5], color=["orange", "teal", "gray"]); plt.ylim(0, 1.05); plt.title("Advanced 5: AP by object scale"); plt.ylabel("AP"); plt.show() # visualize scale-specific detection quality.

▶ What you'll see: small-object AP is visibly lower than large-object AP.

👀 Takeaway: FPN should be evaluated by object scale, not only by one aggregate detection number.